In [32]:
#PROJECT NAME: GROUP DNA
#STUDENT NAME: Sri Karthika Priya
#BATCH: JUNE 2026

#Feature:1 The Chat Parser

with open('/content/hostel_bois.txt','r',encoding='utf-8') as f:   #Open and read the file
    content = f.read()

lines = content.split('\n')

deleted_count = 0     #Declaraing the variables
media_count = 0
systemmsg_count = 0

participants = set()    #Set automatically removes duplicate
messages = []

def encrypted_data(rest):
    global deleted_count, media_count, systemmsg_count #Made global so that can be used out of function

    if "deleted" in rest:
        deleted_count+=1
    elif "media omitted" in rest:
        media_count+=1
    else:
        systemmsg_count+=1


def parse_data(timestamp, rest):
    global messages

    name, message = rest.split(":", 1)
    participants.add(name)
    data = {
        "Timestamp": timestamp,
        "Participant": name,
        "Message": message
    }

    messages.append(data)   #Adds the data to empty message list


for line in lines:     # for loop

    if line == "":
        continue
    parts1 = line.split("-", 1)
    if len(parts1) != 2:
        systemmsg_count += 1
        continue
    timestamp, rest = parts1
    parts2 = rest.split(":", 1)
    if len(parts2) != 2:
        encrypted_data(rest)
    else:
        parse_data(timestamp, rest)


count_msg = len(messages)
count_names = len(participants)

print(
    f"Successfully parsed {count_msg} messages from "
    f"{count_names} participants, skipped "
    f"{systemmsg_count} system messages, "
    f"{media_count} media omitted, "
    f"{deleted_count} deleted messages"
)

Successfully parsed 2973 messages from 6 participants, skipped 4 system messages, 0 media omitted, 0 deleted messages


In [33]:
#Feature:2 Group Overview

#Arithmetic for duration
def Time_duration(timestamp):
    first = timestamp[0]
    last = timestamp[-1]
    datef, timef = first.split(",",1)
    datel, timel = last.split(",",1)
    dayf, monthf, yearf = map(int, datef.split("/"))
    dayl, monthl, yearl = map(int, datel.split("/"))

    days_in_month = [31,28,31,30,31,30,31,31,30,31,30,31]

    def total_days(day, month, year):
        total = year * 365
        for i in range(month-1):
            total += days_in_month[i]
        total += day
        return total

    start = total_days(dayf, monthf, yearf)
    end = total_days(dayl, monthl, yearl)

    duration = end - start
    return duration


timestamp = []            #Foolproofing Timestamp for duration

for line in lines:
    if " - " in line:
        date_time = line.split(" - ", 1)[0]
        timestamp.append(date_time)

duration=Time_duration(timestamp)      #calling the defined function

#Messages per person
message_count = {}    #Creating a empty Dictionary for per-person count

for msg in messages:         # Dictionary[Key]=value
    sender = msg["Participant"]
    if sender not in message_count:
        message_count[sender] = 1
    else:
        message_count[sender] += 1

sorted_msg = sorted(message_count.items(),      #Sorted with key
                    key=lambda x: x[1],
                    reverse=True)

#Formatting output
print("="*50)
print("GROUP OVERVIEW")
print("="*50)
print(f"Group         :Hostel Bois")
print(f"Duration      :{duration} days")
print(f"Total Messages:{len(messages)}")
print(f"Participants  :{len(participants)}\n")


print(f"MESSAGES PER PERSON")
for name, count in sorted_msg:
    print(f"{name}: {count}")



GROUP OVERVIEW
Group         :Hostel Bois
Duration      :59 days
Total Messages:2973
Participants  :6

MESSAGES PER PERSON
 Rahul: 911
 Priya: 640
 Neha: 584
 Aman: 466
 Karan: 348
 Vikas: 24


In [34]:
#Feature:3 Most Active Day and Hour
def busiest_day_hour(messages):
    date_count = {}
    hour_count = {}
    for msg in messages:
        timestamp = msg["Timestamp"]
        date, time = timestamp.split(",", 1)
        if date in date_count:      #Count dates
            date_count[date] += 1
        else:
            date_count[date] = 1
        hour = int(time.strip().split(":")[0])
        if hour in hour_count:   #Count hours
                    hour_count[hour] += 1
        else:
            hour_count[hour] = 1
    busiest_day = max(date_count, key=date_count.get)
    busiest_hour = max(hour_count, key=hour_count.get)
    print("\nBusiest day :", busiest_day,
          f"({date_count[busiest_day]} messages)")
    print("Busiest hour:",
          f"{busiest_hour:02d}:00 - {(busiest_hour + 1) % 24:02d}:00",
          f"({hour_count[busiest_hour]} messages)")

busiest_day_hour(messages)


Busiest day : 04/05/24 (73 messages)
Busiest hour: 18:00 - 19:00 (229 messages)


In [35]:
# Feature:4 Activity Heatmap

import numpy as np

participants_list = sorted(list(participants))
nrow = len(participants_list)
heatmap = np.zeros((nrow,24),dtype=int)
                                             # Integer indexing
for msg in messages:
    timestamp = msg["Timestamp"]
    sender = msg["Participant"]

    date, time = timestamp.split(",",1)
    hour, minute = time.split(":",1)

    hour = int(hour.strip())

    row = participants_list.index(sender)

    heatmap[row][hour] += 1

sq_heatmap = heatmap.reshape(nrow,8,3)     #Slicing
sq_heatmap = sq_heatmap.sum(axis=2)

normalized_heatmap = np.zeros((nrow,8))    #Normalisation
for i in range(nrow):
    max_value = np.max(sq_heatmap[i])
    if max_value != 0:
        normalized_heatmap[i] = sq_heatmap[i] / max_value

print("\nACTIVITY HEATMAP (messages by 3-hour block)\n")          #visually rendered heatmap
print("          00 03 06 09 12 15 18 21")

for i in range(nrow):
    print(f"{participants_list[i]:10}", end="")
    for value in normalized_heatmap[i]:
        if value == 0:
            print(" . ", end="")
        elif value <= 0.25:
            print(" ░ ", end="")
        elif value <= 0.50:
            print(" ▒ ", end="")
        elif value <= 0.75:
            print(" ▓ ", end="")
        else:
            print(" █ ", end="")
    print()


ACTIVITY HEATMAP (messages by 3-hour block)

          00 03 06 09 12 15 18 21
 Aman      █  █  .  .  ░  ░  ░  ▒ 
 Karan     .  .  ░  ▓  █  █  █  ▒ 
 Neha      .  ░  ▒  █  ▓  ▓  █  ▓ 
 Priya     .  .  ▒  █  ▓  ▓  ▓  ▒ 
 Rahul     ░  ░  ▒  ▒  ▓  █  █  █ 
 Vikas     .  .  █  ▒  █  █  █  █ 


In [36]:
#Feature:5 Top Words

words_dict = {}

common_words = {
    "is","i","the","a","and","or","to","of","in","on","for","was","this","that",
    "you","so","am","today","tomorrow","at","they","we","when","how","what","he",
    "she","my","his","her","have","had","has","me","it","all"
}

for msg in messages:       #loops over messages
    text = msg["Message"]
    words = text.split()     #To tokenize words
    for word in words:
        word = word.lower()       #To normalise
        word = word.strip(".,:;!?@#$&()[]{}-")   #to remove punctuation characters
        if word==" " or word in common_words:
          continue

        if word not in words_dict:
            words_dict[word] = 1
        else:
            words_dict[word] += 1

sorted_words = sorted(words_dict.items(),    #Sorted
                      key=lambda x: x[1],
                      reverse=True)

print("THIS GROUP'S FAVOURITE WORDS")       #Prints top five words
for i in range(5):
    print(sorted_words[i])


THIS GROUP'S FAVOURITE WORDS
('guys', 318)
('about', 274)
('hai', 268)
('just', 208)
('which', 202)


In [37]:
#Feature: 6 Response Time and Silent Streaks
from datetime import datetime

response_speed = {}           #Create empty Dictionaries
silent_streak = {}

for person in participants:
    response_speed[person] = []
    silent_streak[person] = []
previous_person = None
previous_time = None
last_time = {}

for msg in messages:
    person = msg["Participant"]
    timestamp = msg["Timestamp"]
    current_time = datetime.strptime(timestamp, "%d/%m/%y, %H:%M ")     #Datetime parsing
    if previous_time is not None:             #Response Time-Arithmetic on time differences
        gap = (current_time - previous_time).total_seconds()
        if person != previous_person:
            response_speed[person].append(gap)
    if person in last_time:                #Silent Streak
        gap = (current_time - last_time[person]).total_seconds()
        silent_streak[person].append(gap)

    last_time[person] = current_time       #Updating the Values
    previous_person = person
    previous_time = current_time


print("RESPONSE PATTERNS\n")        #Response time output
fastest_person = ""
slowest_person = ""
fastest_time = None
slowest_time = None
for person in response_speed:
    if len(response_speed[person]) > 0:
        avg = sum(response_speed[person]) / len(response_speed[person])
        if fastest_time is None or avg < fastest_time:
            fastest_time = avg
            fastest_person = person
        if slowest_time is None or avg > slowest_time:
            slowest_time = avg
            slowest_person = person

print(f"Fastest Replier : {fastest_person} (avg {fastest_time/60:.1f} minutes)")
print(f"Slowest Replier : {slowest_person} (avg {slowest_time/3600:.1f} hours)")

print("\nLONGEST SILENT STREAKS\n")        #Silent streak output

streaks_list = []
for person in silent_streak:
    if len(silent_streak[person]) > 0:
        max_gap = max(silent_streak[person])
        streaks_list.append((person, max_gap))
    else:
        streaks_list.append((person, -1))

sorted_streaks = sorted(streaks_list, key=lambda x: x[1], reverse=True)   #Sort the list

for person, max_gap in sorted_streaks:       #Print the sorted list
    if max_gap == -1:
        print(person, ": no data")
    else:
        if max_gap >= 86400:                   #convert seconds to days or hours
            print(f"{person} : {max_gap/86400:.1f} days")
        elif max_gap >= 3600:
            print(f"{person} : {max_gap/3600:.1f} hours")
        else:
            print(f"{person} : {max_gap/60:.1f} minutes")

RESPONSE PATTERNS

Fastest Replier :  Rahul (avg 37.7 minutes)
Slowest Replier :  Aman (avg 1.0 hours)

LONGEST SILENT STREAKS

 Vikas : 12.2 days
 Neha : 1.2 days
 Rahul : 1.0 days
 Aman : 21.9 hours
 Karan : 20.7 hours
 Priya : 15.8 hours


In [38]:
#Feature: 7 Personality Archetype Detection

#THE SPAMMER Archetype
def spammer(messages):
    message_count = {}
    burst_count = {}
    previous_sender = None
    burst_length = 0
    for msg in messages:
        sender = msg["Participant"]
        if sender not in message_count:
            message_count[sender] = 0
            burst_count[sender] = []
        message_count[sender] += 1

        if sender == previous_sender:
            burst_length += 1
        else:
            if previous_sender is not None:
                burst_count[previous_sender].append(burst_length)
            burst_length = 1
        previous_sender = sender

    if previous_sender is not None:
        burst_count[previous_sender].append(burst_length)

    return burst_count

#THE GROUP MOM Archetype
def group_mom(messages):
  caring_wordcount={}
  caring_word=['okay','safe','eat','sleep','takecare',
               'are you','please','reminder','drink water',
               "don't forget"]
  for msg in messages:
    sender=msg["Participant"]
    if sender not in caring_wordcount:
      caring_wordcount[sender]=0

    text=msg["Message"].lower()    #normalize
    for word in caring_word:
      count=text.count(word)
      caring_wordcount[sender]+=count
  return caring_wordcount

#THE NIGHT OWL Archetype
def night_owl(messages):
    nightmsg_count = {}        #Create empty dict
    total_msg_count = {}
    for msg in messages:
        sender = msg["Participant"]
        if sender not in nightmsg_count:
            nightmsg_count[sender] = 0
            total_msg_count[sender] = 0
    for msg in messages:               #Two loops prevent common tracking bug
        timestamp = msg["Timestamp"]
        sender = msg["Participant"]
        total_msg_count[sender] += 1
        time_obj = datetime.strptime(timestamp, "%d/%m/%y, %H:%M ")
        if time_obj.hour == 23 or time_obj.hour < 5:
            nightmsg_count[sender] += 1

    night_percentage = {}    #Calculate percentage
    for sender in nightmsg_count:
        if total_msg_count[sender] > 0:
            percentage = (nightmsg_count[sender] / total_msg_count[sender]) * 100
            night_percentage[sender] = round(percentage, 1)  # Keeps it to 1 decimal place
        else:
            night_percentage[sender] = 0.0

    return night_percentage

#THE STORYTELLER Archetype
def storyteller(messages):
    total_words = {}      #Create empty dict
    total_msg_count = {}
    for msg in messages:
        sender = msg["Participant"]
        text = msg["Message"]
        words = len(text.split())
        if sender not in total_words:
            total_words[sender] = 0
            total_msg_count[sender] = 0
        total_words[sender] += words
        total_msg_count[sender] += 1

    avg_words = {}
    for sender in total_words:
        if total_msg_count[sender] > 0:
            avg_words[sender] = total_words[sender] / total_msg_count[sender]
        else:
            avg_words[sender] = 0.0
    return avg_words

#THE DRAMA QUEEN Archetype
def drama_queen(messages):
    drama_msg_count = {}    #create empty dict
    total_msg_count = {}
    for msg in messages:
        sender = msg["Participant"]
        text = msg["Message"].strip()
        if sender not in drama_msg_count:
            drama_msg_count[sender] = 0
            total_msg_count[sender] = 0

        total_msg_count[sender] += 1

        is_uppercase = len(text) >= 3 and text.isupper()   #Condition:1 Check for uppercase

        has_exclamations = text.count('!') >= 2     #Condition:2 More than 2 exclamatory marks

        if is_uppercase or has_exclamations:
            drama_msg_count[sender] += 1

    drama_percentage = {}        #To calculate percentage
    for sender in drama_msg_count:
        if total_msg_count[sender] > 0:
            drama_percentage[sender] = (drama_msg_count[sender] / total_msg_count[sender]) * 100
        else:
            drama_percentage[sender] = 0.0
    return drama_percentage

#THE GHOST Archetype
def ghost(messages):
    user_active_days = {}
    all_days = set()
    for msg in messages:
        sender = msg["Participant"]
        date_str = msg["Timestamp"].split(',')[0].strip()   #taking only the date part from timestamp
        all_days.add(date_str)

        if sender not in user_active_days:
            user_active_days[sender] = set()
        user_active_days[sender].add(date_str)

    total_days = len(all_days)
    silent_percentage = {}

    for sender in user_active_days:
        active_days_count = len(user_active_days[sender])
        silent_days_count = total_days - active_days_count
        if total_days > 0:
            silent_percentage[sender] = (silent_days_count / total_days) * 100
        else:
            silent_percentage[sender] = 0.0

    return silent_percentage

#THE COMEDIAN Archetype
def comedian(messages):
    funny_count = {}
    total_msg_count = {}
    slangs = ['lol', 'lmao', 'haha', 'rofl', 'lmfao']
    for msg in messages:
        sender = msg["Participant"]
        text = msg["Message"].lower()     #To normalize
        if sender not in funny_count:
            funny_count[sender] = 0
            total_msg_count[sender] = 0

        total_msg_count[sender] += 1

        for slang in slangs:
            funny_count[sender] += text.count(slang)

    funny_percentage = {}          #Calculate percentage
    for sender in funny_count:
        if total_msg_count[sender] > 0:
            funny_percentage[sender] = (funny_count[sender] / total_msg_count[sender]) * 100
        else:
            funny_percentage[sender] = 0.0
    return funny_percentage

#THE QUESTION MASTER Archetype
def question_master(messages):
    question_count = {}           #Create empty dict
    total_msg_count = {}
    for msg in messages:
        sender = msg["Participant"]
        text = msg["Message"].strip()
        if sender not in question_count:
            question_count[sender] = 0
            total_msg_count[sender] = 0

        total_msg_count[sender] += 1

        if text.endswith('?'):
            question_count[sender] += 1

    question_percentage = {}    #calculate percentage
    for sender in question_count:
        if total_msg_count[sender] > 0:
            question_percentage[sender] = (question_count[sender] / total_msg_count[sender]) * 100
        else:
            question_percentage[sender] = 0.0
    return question_percentage

#All the separate functions containing condtions for archetype are brought together

def build_scoreboard(messages):

    spammer_raw = spammer(messages)        #Calling all the feature functions
    spammer_metrics = {}

    for person in spammer_raw:
        if len(spammer_raw[person]) > 0:
            spammer_metrics[person] = sum(spammer_raw[person]) / len(spammer_raw[person])
        else:
            spammer_metrics[person] = 0

    group_mom_metrics = group_mom(messages)
    night_owl_metrics = night_owl(messages)
    storyteller_metrics = storyteller(messages)
    drama_queen_metrics = drama_queen(messages)
    ghost_metrics = ghost(messages)
    comedian_metrics = comedian(messages)
    question_master_metrics = question_master(messages)

    participants = list(set(msg["Participant"] for msg in messages))     #Unique participants


    scoreboard = {}          #Scoreboard for archetypes

    for person in participants:
        scoreboard[person] = {
            "SPAMMER": spammer_metrics.get(person, 0),
            "GROUP MOM": group_mom_metrics.get(person, 0),
            "NIGHT OWL": night_owl_metrics.get(person, 0),
            "STORYTELLER": storyteller_metrics.get(person, 0),
            "DRAMA QUEEN": drama_queen_metrics.get(person, 0),
            "GHOST": ghost_metrics.get(person, 0),
            "COMEDIAN": comedian_metrics.get(person, 0),
            "QUESTION MASTER": question_master_metrics.get(person, 0)
        }

    person_rankings = {}

    for person in participants:
        sorted_archetypes = sorted(
            scoreboard[person].items(),
            key=lambda x: x[1],
            reverse=True
        )

        person_rankings[person] = []

        for archetype, score in sorted_archetypes:
            person_rankings[person].append(archetype)

    participants_by_dominance = sorted(                   #Sorting the scoreboard for the highest
        participants,
        key=lambda person: max(scoreboard[person].values()),
        reverse=True
    )

    assignments = {}                #Unique archetype for individual person
    used_archetypes = set()

    for person in participants_by_dominance:
      for archetype in person_rankings[person]:
        if archetype not in used_archetypes:
            assignments[person] = archetype
            used_archetypes.add(archetype)
            break

    print("\nPERSONALITY ARCHETYPES\n")         #Formatting Output

    for person, archetype in assignments.items():

        score = scoreboard[person][archetype]

        if archetype == "SPAMMER":
            detail = f"(avg {score:.1f} msgs in a row)"
        elif archetype == "GROUP MOM":
            detail = f"({int(score)} caring keywords)"
        elif archetype == "NIGHT OWL":
            detail = f"({score:.1f}% msgs after 11 PM)"
        elif archetype == "STORYTELLER":
            detail = f"(avg {score:.1f} words/msg)"
        elif archetype == "DRAMA QUEEN":
            detail = f"({score:.1f}% dramatic msgs)"
        elif archetype == "GHOST":
            detail = f"(silent {score:.1f}% of days)"
        elif archetype == "COMEDIAN":
            detail = f"({score:.1f}% funny slangs)"
        elif archetype == "QUESTION MASTER":
            detail = f"({score:.1f}% question msgs)"
        print(f"{person:<12} -> THE {archetype:<16} {detail}")

    return assignments

In [41]:
#Feature: 8 The Final Report

#Feature:2 Group Overview
print("="*50)
print("GROUP OVERVIEW")
print("="*50)
print(f"Group         :Hostel Bois")
print(f"Duration      :{duration} days")
print(f"Total Messages:{len(messages)}")
print(f"Participants  :{len(participants)}\n")


print(f"MESSAGES PER PERSON")
for name, count in sorted_msg:
    print(f"{name}: {count}")

#Feature:3
busiest_day_hour(messages)

#Feature:4
print("\nACTIVITY HEATMAP (messages by 3-hour block)\n")
print("          00 03 06 09 12 15 18 21")

for i in range(nrow):
    print(f"{participants_list[i]:10}", end="")
    for value in normalized_heatmap[i]:
        if value == 0:
            print(" . ", end="")
        elif value <= 0.25:
            print(" ░ ", end="")
        elif value <= 0.50:
            print(" ▒ ", end="")
        elif value <= 0.75:
            print(" ▓ ", end="")
        else:
            print(" █ ", end="")
    print()

#Feature: 5
print("\nTHIS GROUP'S FAVOURITE WORDS")
print("-" * 40)  # separator line
for i in reversed(range(5)):      #sorted with reverse order
    word, count = sorted_words[i]
    bar_length = count // 10      #horizontal bar length
    bar_visual = "█" * bar_length
    print(f"{word:10} : {bar_visual} ({count})")

#feature: 6
print("\nRESPONSE PATTERNS\n")        #Response time output
fastest_person = ""
slowest_person = ""
fastest_time = None
slowest_time = None
for person in response_speed:
    if len(response_speed[person]) > 0:
        avg = sum(response_speed[person]) / len(response_speed[person])
        if fastest_time is None or avg < fastest_time:
            fastest_time = avg
            fastest_person = person
        if slowest_time is None or avg > slowest_time:
            slowest_time = avg
            slowest_person = person
print(f"Fastest Replier : {fastest_person} (avg {fastest_time/60:.1f} minutes)")
print(f"Slowest Replier : {slowest_person} (avg {slowest_time/3600:.1f} hours)")

print("\nLONGEST SILENT STREAKS\n")
streaks_list = []
for person in silent_streak:
    if len(silent_streak[person]) > 0:
        max_gap = max(silent_streak[person])
        streaks_list.append((person, max_gap))
    else:
        streaks_list.append((person, -1))

sorted_streaks = sorted(streaks_list, key=lambda x: x[1], reverse=True)   #Sort the list

for person, max_gap in sorted_streaks:       #Print the sorted list
    if max_gap == -1:
        print(person, ": no data")
    else:
        if max_gap >= 86400:                   #convert seconds to days or hours
            print(f"{person} : {max_gap/86400:.1f} days")
        elif max_gap >= 3600:
            print(f"{person} : {max_gap/3600:.1f} hours")
        else:
            print(f"{person} : {max_gap/60:.1f} minutes")

#Feature: 7
build_scoreboard(messages)

print("="*50)
print("Built with Python + NumPy")
print("="*50)



GROUP OVERVIEW
Group         :Hostel Bois
Duration      :59 days
Total Messages:2973
Participants  :6

MESSAGES PER PERSON
 Rahul: 911
 Priya: 640
 Neha: 584
 Aman: 466
 Karan: 348
 Vikas: 24

Busiest day : 04/05/24 (73 messages)
Busiest hour: 18:00 - 19:00 (229 messages)

ACTIVITY HEATMAP (messages by 3-hour block)

          00 03 06 09 12 15 18 21
 Aman      █  █  .  .  ░  ░  ░  ▒ 
 Karan     .  .  ░  ▓  █  █  █  ▒ 
 Neha      .  ░  ▒  █  ▓  ▓  █  ▓ 
 Priya     .  .  ▒  █  ▓  ▓  ▓  ▒ 
 Rahul     ░  ░  ▒  ▒  ▓  █  █  █ 
 Vikas     .  .  █  ▒  █  █  █  █ 

THIS GROUP'S FAVOURITE WORDS
----------------------------------------
which      : ████████████████████ (202)
just       : ████████████████████ (208)
hai        : ██████████████████████████ (268)
about      : ███████████████████████████ (274)
guys       : ███████████████████████████████ (318)

RESPONSE PATTERNS

Fastest Replier :  Rahul (avg 37.7 minutes)
Slowest Replier :  Aman (avg 1.0 hours)

LONGEST SILENT STREAKS

 Vikas : 12.2